# 02 - Multilingual Embeddings and Confound Diagnostics

                This notebook computes or loads article embeddings. The default model follows the revised plan and uses `Qwen/Qwen3-Embedding-0.6B`, with a multilingual sentence-transformer fallback if the Qwen model cannot be loaded locally. Embeddings are cached, and all downstream notebooks read the cached `.npy` files.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

## Configuration

In [ ]:
MAIN_EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
FALLBACK_EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
ALT_EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

TEXT_COLUMN = "text_for_embedding"
MAX_CHARS = 6000
BATCH_SIZE = 16
COMPUTE_ALT_EMBEDDINGS = False
ALLOW_FALLBACK_MODEL = True

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
embedding_articles = articles[articles["corpus_inclusion_core"].fillna(False)].copy()
embedding_articles = embedding_articles[embedding_articles[TEXT_COLUMN].map(lambda x: len(compact_ws(x)) >= 120)].copy()
embedding_articles = embedding_articles.sort_values(["year", "article_id"]).reset_index(drop=True)
print(f"Embedding records: {len(embedding_articles):,}")

## Compute or Load Embeddings

In [ ]:
from pathlib import Path

def load_sentence_transformer(model_name):
    from sentence_transformers import SentenceTransformer
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
        elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
    except Exception:
        device = "cpu"
    try:
        model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    except TypeError:
        model = SentenceTransformer(model_name, device=device)
    return model

def compute_or_load_embeddings(df, model_name, output_path, index_path, batch_size=BATCH_SIZE):
    output_path = Path(output_path)
    index_path = Path(index_path)
    if output_path.exists() and index_path.exists():
        matrix = np.load(output_path)
        index = pd.read_csv(index_path)
        cached_model = index["embedding_model"].iloc[0] if "embedding_model" in index.columns and len(index) else model_name
        print(f"Loaded cached embeddings: {output_path.relative_to(ROOT)} {matrix.shape}")
        return matrix, index, cached_model

    texts = df[TEXT_COLUMN].map(lambda x: compact_ws(x)[:MAX_CHARS]).tolist()
    model = load_sentence_transformer(model_name)
    matrix = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    matrix = np.asarray(matrix, dtype=np.float32)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(output_path, matrix)
    index = df[["article_id", "year", "language", "title", "text_length_chars", "text_length_words"]].copy()
    index["embedding_row"] = np.arange(len(index))
    index["embedding_model"] = model_name
    index["text_column"] = TEXT_COLUMN
    index["max_chars"] = MAX_CHARS
    write_csv(index, index_path)
    return matrix, index, model_name

main_path = ROOT / "outputs/models/article_embeddings_main.npy"
main_index_path = ROOT / "outputs/tables/article_embedding_index.csv"

try:
    embeddings, embedding_index, model_used = compute_or_load_embeddings(
        embedding_articles,
        MAIN_EMBEDDING_MODEL,
        main_path,
        main_index_path,
    )
except Exception as exc:
    if not ALLOW_FALLBACK_MODEL:
        raise
    print(f"Main model failed: {exc}")
    print(f"Falling back to {FALLBACK_EMBEDDING_MODEL}")
    embeddings, embedding_index, model_used = compute_or_load_embeddings(
        embedding_articles,
        FALLBACK_EMBEDDING_MODEL,
        main_path,
        main_index_path,
    )

runs_path = ROOT / "outputs/tables/embedding_model_runs.csv"
run_row = pd.DataFrame(
    [
        {
            "embedding_slot": "main",
            "requested_model": MAIN_EMBEDDING_MODEL,
            "model_used": model_used,
            "n_articles": embeddings.shape[0],
            "n_dimensions": embeddings.shape[1],
            "text_column": TEXT_COLUMN,
            "max_chars": MAX_CHARS,
        }
    ]
)
write_csv(run_row, runs_path)
print(embeddings.shape)
display(embedding_index.head())

## Optional Alternative Embedding Model

In [ ]:
if COMPUTE_ALT_EMBEDDINGS:
    alt_embeddings, alt_index, alt_model_used = compute_or_load_embeddings(
        embedding_articles,
        ALT_EMBEDDING_MODEL,
        ROOT / "outputs/models/article_embeddings_alt.npy",
        ROOT / "outputs/tables/article_embedding_index_alt.csv",
    )
    alt_row = pd.DataFrame(
        [
            {
                "embedding_slot": "alt",
                "requested_model": ALT_EMBEDDING_MODEL,
                "model_used": alt_model_used,
                "n_articles": alt_embeddings.shape[0],
                "n_dimensions": alt_embeddings.shape[1],
                "text_column": TEXT_COLUMN,
                "max_chars": MAX_CHARS,
            }
        ]
    )
    previous = pd.read_csv(runs_path) if runs_path.exists() else pd.DataFrame()
    write_csv(pd.concat([previous, alt_row], ignore_index=True), runs_path)
else:
    print("Alternative embeddings are configured but not computed. Set COMPUTE_ALT_EMBEDDINGS = True for the robustness run.")

## Language and Length Diagnostics in the Embedding Space

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

try:
    import umap
except ImportError as exc:
    raise ImportError("Install umap-learn to create diagnostic maps.") from exc

sns.set_theme(style="white")
diagnostic_mapper = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.10, metric="cosine", random_state=42)
diag_xy = diagnostic_mapper.fit_transform(embeddings)
diag = embedding_index.copy()
diag["umap_x"] = diag_xy[:, 0]
diag["umap_y"] = diag_xy[:, 1]
write_csv(diag, ROOT / "outputs/tables/embedding_diagnostic_coordinates.csv")

language_summary = (
    diag.groupby("language", dropna=False)
    .agg(
        n_articles=("article_id", "nunique"),
        mean_text_length_words=("text_length_words", "mean"),
        median_text_length_words=("text_length_words", "median"),
    )
    .reset_index()
)
write_csv(language_summary, ROOT / "outputs/tables/language_embedding_input_summary.csv")

fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=diag, x="umap_x", y="umap_y", hue="language", s=18, alpha=0.75, ax=ax)
ax.set_title("Language in embedding projection")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_language_in_embedding_space.png", dpi=220)
save_caption(ROOT, "fig_language_in_embedding_space.png", "Exploratory UMAP projection colored by article language; coordinates are not interpreted as metric distances.")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(diag["umap_x"], diag["umap_y"], c=np.log1p(diag["text_length_words"]), s=18, alpha=0.75, cmap="viridis")
ax.set_title("Text length in embedding projection")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
fig.colorbar(scatter, ax=ax, label="log(1 + words)")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_text_length_in_embedding_space.png", dpi=220)
save_caption(ROOT, "fig_text_length_in_embedding_space.png", "Exploratory UMAP projection colored by input text length.")
plt.show()
display(language_summary)